# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03"
)

march = pd.read_parquet(march_path)

print("Rows:", len(march))
print("Columns:", len(march.columns))
print("Date range:", march["report_date"].min(), "to", march["report_date"].max())

Rows: 9841378
Columns: 30
Date range: 2026-03-01 to 2026-03-31


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will rank content-refresh candidates using a simple opportunity score based on search visibility and average search position. Pages with meaningful search impressions and weaker average positions receive higher scores because they have observable search visibility and potential room for improvement.

The rule is intended for human decision-support, not as an automatic instruction to refresh a page. It does not use future performance or label-derived fields.

### Reason codes

- `VISIBLE_LOW_POSITION` — the page has search impressions and an average position between 21 and 50.
- `VISIBLE_POSITION_OPPORTUNITY` — the page has search impressions and an average position between 11 and 20.
- `LOW_PRIORITY` — the page does not meet either opportunity condition.

In [2]:
# Signal 1: Search visibility / impressions

baseline_signal = march[
    march["gsc_data_available"] == True
].copy()

baseline_signal["impression_bucket"] = pd.cut(
    baseline_signal["gsc_impressions"],
    bins=[-1, 0, 99, 499, float("inf")],
    labels=[
        "0",
        "1-99",
        "100-499",
        "500+"
    ]
)

impression_table = (
    baseline_signal["impression_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("impression_bucket")
    .reset_index(name="n")
)

print("Signal 1: GSC impressions")
print(impression_table)

Signal 1: GSC impressions
  impression_bucket        n
0                 0        0
1              1-99  2972453
2           100-499   537157
3              500+   101451


**Verdict: CONFIRMED**

Search impressions provide an observed measure of search visibility. The distribution shows that some content has enough impressions to distinguish meaningful visibility from very low-volume observations. I therefore use 100 impressions as a minimum threshold for the refresh-opportunity rule.

In [3]:
# Signal 2: Average search position

position_signal = march[
    (march["gsc_data_available"] == True)
    & (march["gsc_avg_position"] > 0)
].copy()

position_signal["position_bucket"] = pd.cut(
    position_signal["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=[
        "1-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

position_table = (
    position_signal["position_bucket"]
    .value_counts()
    .sort_index()
    .rename_axis("position_bucket")
    .reset_index(name="n")
)

print("Signal 2: GSC average position")
print(position_table)

Signal 2: GSC average position
  position_bucket        n
0            1-10  2020295
1           11-20   519223
2           21-50   631491
3             50+   276863


**Verdict: CONFIRMED**

Average search position provides an observed signal of where content appears in search results. Positions 11-20 and 21-50 represent measurable visibility with potential room for improvement, so I use these ranges as directional opportunity buckets rather than proof that a refresh will work.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

baseline = march.copy()

# Position = 0 means there is no usable average position.
baseline["position_clean"] = baseline["gsc_avg_position"].replace(0, np.nan)

# Only use rows where GSC data is available.
baseline = baseline[
    baseline["gsc_data_available"] == True
].copy()

# Start with zero score.
baseline["score"] = 0

# Search visibility:
# Pages need at least 100 impressions to qualify for an opportunity score.
baseline.loc[
    baseline["gsc_impressions"] >= 100,
    "score"
] += 1

# Stronger visibility.
baseline.loc[
    baseline["gsc_impressions"] >= 500,
    "score"
] += 1

# Position opportunity.
baseline.loc[
    baseline["position_clean"].between(11, 20)
    & (baseline["gsc_impressions"] >= 100),
    "score"
] += 2

baseline.loc[
    baseline["position_clean"].between(21, 50)
    & (baseline["gsc_impressions"] >= 100),
    "score"
] += 3

# ONE reason code.
baseline["reason_code"] = np.select(
    [
        (baseline["gsc_impressions"] >= 100)
        & baseline["position_clean"].between(21, 50),

        (baseline["gsc_impressions"] >= 100)
        & baseline["position_clean"].between(11, 20)
    ],
    [
        "VISIBLE_LOW_POSITION",
        "VISIBLE_POSITION_OPPORTUNITY"
    ],
    default="LOW_PRIORITY"
)

# Action must agree with the reason.
baseline["action"] = np.where(
    baseline["reason_code"].isin([
        "VISIBLE_LOW_POSITION",
        "VISIBLE_POSITION_OPPORTUNITY"
    ]),
    "REVIEW_FOR_REFRESH",
    "MONITOR"
)

print("Rows scored:", len(baseline))

print("\nReason-code counts:")
print(baseline["reason_code"].value_counts())

print("\nAction counts:")
print(baseline["action"].value_counts())

Rows scored: 3611061

Reason-code counts:
reason_code
LOW_PRIORITY                    3433571
VISIBLE_LOW_POSITION             116779
VISIBLE_POSITION_OPPORTUNITY      60711
Name: count, dtype: int64

Action counts:
action
MONITOR               3433571
REVIEW_FOR_REFRESH     177490
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked queue

The score ranks rows using only information that would be available at the decision moment. I use GSC impressions and average position as the observed signals. Rows are sorted from highest to lowest score, and each row receives an action label and one reason code. The resulting queue is written to work/outputs/baseline_action_score.csv.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Rank content items by score.
# Impressions are used as a tie-breaker when scores are equal.

baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

# Give each row a rank.
baseline["rank"] = baseline.index + 1

# Select the columns needed for the action queue.
queue = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
].copy()

# Create the output directory if it does not already exist.
import os

os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV file.
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue written successfully.")
print("Rows:", len(queue))
print("File: work/outputs/baseline_action_score.csv")

print("\nTop 20:")
print(queue.head(20).to_string(index=False))

Queue written successfully.
Rows: 3611061
File: work/outputs/baseline_action_score.csv

Top 20:
 rank          client_hash_id          content_hash_id report_date  gsc_impressions  gsc_clicks  gsc_avg_position  score          reason_code             action
    1 client_23a62021009f63c4 content_e6df0936699f5b8f  2026-03-31            14682         269         25.035826      5 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
    2 client_23a62021009f63c4 content_36e53e9c707674fc  2026-03-09             9409           1         32.640344      5 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
    3 client_23a62021009f63c4 content_3df3f32f3fd58dea  2026-03-10             9300          14         22.399032      5 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
    4 client_23a62021009f63c4 content_36e53e9c707674fc  2026-03-11             9285          16         32.510609      5 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
    5 client_23a62021009f63c4 content_3df3f32f3fd58dea  2026-03-12             9274          10    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

Top-20 review: The highest-ranked rows are primarily marked REVIEW_FOR_REFRESH because they combine strong observed search visibility with weaker average positions. I use moderate confidence because the dataset does not show content quality, search intent, or business context. These rankings are therefore decision-support rather than proof that a page should definitely be refreshed.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Select the actual top 20 rows from our ranked queue.
top20 = queue.head(20).copy()

# Add a confidence note.
top20["confidence_note"] = (
    "Moderate confidence: strong observed search visibility and "
    "position opportunity, but content quality and search intent are unknown."
)

# Add the required "what would make it wrong" explanation.
top20["what_would_make_it_wrong"] = (
    "It could be wrong if the page has different search intent, "
    "does not need a refresh, or has business/content reasons not "
    "captured by the rule."
)

# Display the required review fields.
print(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)

 rank          content_hash_id             action          reason_code                                                                                                                     confidence_note                                                                                                                      what_would_make_it_wrong
    1 content_e6df0936699f5b8f REVIEW_FOR_REFRESH VISIBLE_LOW_POSITION Moderate confidence: strong observed search visibility and position opportunity, but content quality and search intent are unknown. It could be wrong if the page has different search intent, does not need a refresh, or has business/content reasons not captured by the rule.
    2 content_36e53e9c707674fc REVIEW_FOR_REFRESH VISIBLE_LOW_POSITION Moderate confidence: strong observed search visibility and position opportunity, but content quality and search intent are unknown. It could be wrong if the page has different search intent, does not need a refresh, or has business/content

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

Weak-pick review: Some lower-ranked rows are weak candidates because the rule can assign priority based on limited observed search signals. For example, rows with only 100 impressions can receive a VISIBLE_LOW_POSITION reason even when their average position is relatively poor. This may not represent a meaningful refresh opportunity because the observed volume is still limited. The rule therefore requires human review rather than automatically triggering a content change.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show some examples of potentially weak picks.
# These are candidates for review, not automatic refresh decisions.

# Find examples of potentially weak picks:
# high-ranked review actions with very low impressions.

weak_picks = (
    baseline[
        (baseline["action"] == "REVIEW_FOR_REFRESH")
        & (baseline["gsc_impressions"] < 200)
    ]
    .sort_values(["score", "gsc_impressions"], ascending=[False, True])
    .head(10)
)

print("Examples of potentially weak picks:")
print(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

print("\nLeakage check:")

leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_click",
    "future_impressions"
]

for field in leakage_fields:
    print(
        field + ":",
        "USED" if field in baseline.columns else "NOT USED"
    )

print("\nFuture-window fields used in score: NONE")
print("Label-derived fields used in score: NONE")

Examples of potentially weak picks:
  rank          content_hash_id report_date  gsc_impressions  gsc_avg_position  score          reason_code             action
121598 content_25d3bf0883113206  2026-03-02              100             47.96      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121599 content_f6c2245c6d8cdd3c  2026-03-02              100             22.48      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121600 content_fd4fc2fdac938fa8  2026-03-02              100             47.30      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121601 content_73783bbf4f747249  2026-03-03              100             28.53      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121602 content_f520e8732cd4263e  2026-03-03              100             32.31      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121603 content_c027b95f559d757a  2026-03-03              100             35.56      4 VISIBLE_LOW_POSITION REVIEW_FOR_REFRESH
121604 content_9c2b18340755af25  2026-03-03              100             32.10    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.